# Single-Task Fingerprint Neural Models

## Scientific objective
Train one early-stopped MLP per endpoint using fingerprint inputs, class weighting, scheduling, clipping, checkpoints, and test evaluation.

## Inputs
- Morgan features
- Modeling records
- Training config

## Expected outputs
- `models/neural/*_mlp.pt`
- `results/metrics/single_task_mlp.csv`
- training histories

## Dependencies
PyTorch

## Reproducibility seed
`20260723`. The seed is loaded from `configs/training_config.yaml`; split files and checkpoints are persisted.

## Data and model assumptions
Neural models use the same scaffold partitions as QSAR controls. Missing labels are excluded endpoint by endpoint.

## Validation checks
The executable cells below fail explicitly on missing/inconsistent required artifacts and save machine-readable status records.

## Interpretation of results
Interpret endpoint-level outputs only after checking prevalence, missingness, split integrity, calibration, uncertainty, and applicability-domain coverage. No notebook result is evidence that experimental toxicity testing can be replaced.

## Saved artifacts
Artifacts listed above are written under `data/`, `models/`, `results/`, `figures/`, `tables/`, or `reports/` and are consumed by later notebooks.

## Limitations
A single smoke run is not evidence of superiority. Full evaluation requires repeated seeds and confidence intervals.

## Next notebook
[12_single_task_gnn_models.ipynb](./12_single_task_gnn_models.ipynb)

In [1]:
from pathlib import Path
import os, json, warnings
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks directory")
os.chdir(ROOT)

from toxicity_screening.config import load_configs, execution_profile
from toxicity_screening.utils import set_global_seed, require_paths

CONFIGS = load_configs(ROOT)
PROFILE, PROFILE_CONFIG = execution_profile(CONFIGS)
SEED = int(CONFIGS["training_config"]["seed"])
set_global_seed(SEED)
print({"root": str(ROOT), "profile": PROFILE, "seed": SEED})

{'root': 'D:\\Dropbox\\Work\\Learning\\Python\\toxicity_screening_project', 'profile': 'full', 'seed': 20260723}


In [2]:
import torch
from torch.utils.data import DataLoader
from toxicity_screening.datasets import ArrayDataset
from toxicity_screening.neural_models import FingerprintMLP
from toxicity_screening.training import train_binary_model, resolve_device
from toxicity_screening.metrics import binary_metrics
X=np.load(ROOT/"data/processed/morgan_features.npz")["X"].astype(np.float32)
ids=np.load(ROOT/"data/processed/morgan_features.npz")["molecule_id"].astype(str)
index=pd.DataFrame({"molecule_id":ids,"row":np.arange(len(ids))})
records = pd.read_parquet(
    ROOT / "data/processed/modeling_records.parquet"
).merge(
    index,
    on="molecule_id",
    validate="many_to_one",
)
rows=[]
for endpoint, frame in records[records.label.notna()].groupby("endpoint"):
    parts={p:frame[frame.scaffold_split==p] for p in ["train","validation","test"]}
    cap=PROFILE_CONFIG["sample_cap_per_endpoint"]
    train=parts["train"].sample(min(len(parts["train"]),cap or len(parts["train"])),random_state=SEED)
    loaders={}
    for p,part in {"train":train,"validation":parts["validation"]}.items():
        loaders[p]=DataLoader(ArrayDataset(X[part.row.astype(int)],part.label.to_numpy(float)),batch_size=CONFIGS["training_config"]["batch_size"],shuffle=p=="train")
    model=FingerprintMLP(X.shape[1], **CONFIGS["model_config"]["neural"]["fingerprint_mlp"])
    ytrain=train.label.to_numpy(int); pos_weight=float((ytrain==0).sum()/max(1,(ytrain==1).sum()))
    result=train_binary_model(model,loaders["train"],loaders["validation"],epochs=PROFILE_CONFIG["max_epochs"],patience=PROFILE_CONFIG["patience"],learning_rate=CONFIGS["training_config"]["optimizer"]["learning_rate"],weight_decay=CONFIGS["training_config"]["optimizer"]["weight_decay"],gradient_clip_norm=CONFIGS["training_config"]["gradient_clip_norm"],positive_weight=pos_weight,checkpoint_path=ROOT/f"models/neural/{endpoint}_mlp.pt")
    pd.DataFrame(result.history).to_csv(ROOT/f"results/metrics/{endpoint}_mlp_history.csv",index=False)
    model.eval(); device=resolve_device(); model.to(device)
    xt=torch.as_tensor(X[parts["test"].row.astype(int)],dtype=torch.float32,device=device)
    with torch.no_grad(): prob=torch.sigmoid(model(xt)).cpu().numpy()
    rows.append({"endpoint":endpoint,"model":"fingerprint_mlp",**{k:v for k,v in binary_metrics(parts["test"].label.astype(int),prob).items() if k!="confusion_matrix"}})
mlp_metrics=pd.DataFrame(rows); mlp_metrics.to_csv(ROOT/"results/metrics/single_task_mlp.csv",index=False); display(mlp_metrics)

,endpoint,model,n,positive_prevalence,threshold,roc_auc,pr_auc,mcc,accuracy,balanced_accuracy,...,f1,brier,ece,nll,recall_at_precision_0.80,precision_at_recall_0.80,tn,fp,fn,tp
0,SR-ARE,fingerprint_mlp,847,0.205431,0.5,0.763894,0.501459,0.344748,0.711924,0.705799,...,0.497942,0.200882,0.201735,0.605886,0.114943,0.316027,482,191,53,121
1,SR-ATAD5,fingerprint_mlp,1027,0.064265,0.5,0.727328,0.175134,0.167393,0.650438,0.665051,...,0.200445,0.217430,0.378923,0.627206,0.000000,0.101664,623,338,21,45
2,SR-MMP,fingerprint_mlp,842,0.220903,0.5,0.840156,0.635222,0.487132,0.808789,0.759802,...,0.608273,0.143450,0.120861,0.463486,0.301075,0.436950,556,100,61,125
3,SR-p53,fingerprint_mlp,986,0.085193,0.5,0.762248,0.318745,0.278689,0.862069,0.665505,...,0.346154,0.136860,0.251770,0.446708,0.023810,0.137931,814,88,48,36
4,ames_mutagenicity,fingerprint_mlp,1120,0.547321,0.5,0.852653,0.879032,0.515458,0.748214,0.756682,...,0.743636,0.167450,0.096774,0.506757,0.766721,0.777603,429,78,204,409
5,herg_blockade,fingerprint_mlp,1883,0.507169,0.5,0.807148,0.821754,0.419993,0.707382,0.706034,...,0.734969,0.188668,0.083874,0.562329,0.633508,0.679715,568,360,191,764


### Completion gate
Confirm that the declared artifacts exist before continuing to `12_single_task_gnn_models.ipynb`.